# 01 · ¿Qué es Machine Learning?

**Proyecto:** ML desde cero con datos de salud mental
**Duración:** 60–75 min
**Antes de empezar:** ninguno. Solo pandas básico.

Al terminar vas a poder explicar con tus palabras:
1. La diferencia entre **programar reglas** y **aprender reglas de los datos**.
2. Qué son **X** (features), **y** (objetivo) y una **observación**.
3. Qué es un **baseline** y por qué todo modelo debe superarlo.
4. Cómo se ve un modelo real de scikit-learn por dentro: `fit` → `predict`.

> ⚠️ **Sobre el tema.** Vamos a predecir si una persona **buscó tratamiento de salud mental**.
> Esto es un ejercicio para aprender, **no un diagnóstico**. Un modelo así nunca debería usarse para
> decidir algo sobre una persona real (contratarla, asegurarla, evaluarla). Lo volveremos a discutir
> en cada notebook: en ML con datos de personas, la ética es parte del trabajo, no un extra.

## 1 · La idea central (10 min)

**Programación tradicional:** tú escribes las reglas.

```
reglas  +  datos   →   respuestas
```
Ejemplo: `si temperatura > 38 → "fiebre"`. Tú sabías la regla de antemano.

**Machine Learning:** le das ejemplos con la respuesta correcta y el algoritmo **descubre** las reglas.

```
datos  +  respuestas   →   reglas
```

Analogía: a un niño no le explicas la "regla" de qué es un perro (cuatro patas, pelo, ladra…).
Le muestras muchos perros y muchos gatos, y él solo arma la regla. Eso es ML.

**¿Cuándo vale la pena?** Cuando las reglas son demasiado complejas para escribirlas a mano,
o cuando ni siquiera sabes cuáles son.

## 2 · Conocer los datos (10 min)

Encuesta **OSMI Mental Health in Tech (2014)**: 1.259 personas que trabajan en tecnología
respondieron preguntas sobre su trabajo y su salud mental.

In [2]:
from pathlib import Path

import pandas as pd

datos = pd.read_csv(Path("data") / "survey.csv")

print(datos.shape)
datos.head()

(1259, 27)


,Timestamp,Age,Gender,Country,state,self_employed,family_history,treatment,work_interfere,no_employees,...,leave,mental_health_consequence,phys_health_consequence,coworkers,supervisor,mental_health_interview,phys_health_interview,mental_vs_physical,obs_consequence,comments
0,2014-08-27 11:29:31,37,Female,United States,IL,NaN,No,Yes,Often,6-25,...,Somewhat easy,No,No,Some of them,Yes,No,Maybe,Yes,No,NaN
1,2014-08-27 11:29:37,44,M,United States,IN,NaN,No,No,Rarely,More than 1000,...,Don't know,Maybe,No,No,No,No,No,Don't know,No,NaN
2,2014-08-27 11:29:44,32,Male,Canada,NaN,NaN,No,No,Rarely,6-25,...,Somewhat difficult,No,No,Yes,Yes,Yes,Yes,No,No,NaN
3,2014-08-27 11:29:46,31,Male,United Kingdom,NaN,NaN,Yes,Yes,Often,26-100,...,Somewhat difficult,Yes,Yes,Some of them,No,Maybe,Maybe,No,Yes,NaN
4,2014-08-27 11:30:22,31,Male,United States,TX,NaN,No,No,Never,100-500,...,Don't know,No,No,Some of them,Yes,Yes,Yes,Don't know,No,NaN


Columnas que usaremos hoy:

| Columna | Pregunta de la encuesta |
|---|---|
| `treatment` | ¿Has buscado tratamiento para un problema de salud mental? |
| `family_history` | ¿Tienes antecedentes familiares de enfermedad mental? |
| `care_options` | ¿Sabes qué opciones de atención de salud mental ofrece tu empleador? |
| `benefits` | ¿Tu empleador ofrece beneficios de salud mental? |
| `obs_consequence` | ¿Has visto consecuencias negativas para colegas con problemas de salud mental? |
| `remote_work` | ¿Trabajas de forma remota al menos el 50% del tiempo? |
| `tech_company` | ¿Tu empleador es principalmente una empresa de tecnología? |

## 3 · El vocabulario básico (10 min)

- **Observación**: una fila, o sea, una persona.
- **Objetivo (`y`)**: lo que queremos predecir. Aquí, `treatment`.
- **Features (`X`)**: las columnas que usamos **para** predecir.

Cuando el objetivo es una categoría (Sí/No), el problema se llama **clasificación**.
Si fuera un número (por ejemplo, un salario), sería **regresión**.

### ✍️ Ejercicio 1
1. Crea `y` con la columna `treatment`.
2. Cuenta cuántas personas dijeron `Yes` y cuántas `No`.
3. ¿Qué **porcentaje** dijo `Yes`?

In [3]:
# tu código aquí
y = datos['treatment']
print(y.value_counts())

print(f"{(y == "Yes").mean() * 100:.2f}% dijo YES")

treatment
Yes    637
No     622
Name: count, dtype: int64
50.60% dijo YES


<details><summary><b>Ver solución</b> (inténtalo al menos 5 minutos antes)</summary>

Las clases están casi **equilibradas**: 637 Yes y 622 No (50.6% Yes). Esto importa mucho para
la siguiente sección.

```python
y = datos["treatment"]
print(y.value_counts())
print(f"{(y == 'Yes').mean() * 100:.1f}% dijo Yes")   # 50.6%
```

</details>

## 4 · El baseline: el modelo más tonto posible (5 min)

Antes de construir cualquier modelo, pregúntate: **¿cuánto acierto sin pensar?**

El modelo más tonto responde siempre la clase más común. Aquí, siempre "Yes".
Acertaría el **50.6%** de las veces, casi lo mismo que tirar una moneda.

**Regla de oro:** si tu modelo no supera al baseline, no aprendió nada útil.

> 💡 El baseline depende de los datos. Si el 95% dijera "No", el modelo tonto acertaría el 95%.
> Por eso un "95% de acierto" no significa nada si no lo comparas con el baseline.

## 5 · Tú como programador: escribe una regla a mano (15 min)

Piensa como un humano: ¿qué respuesta crees que se relaciona más con buscar tratamiento?

Una hipótesis razonable: **quien tiene antecedentes familiares tiende a buscar ayuda**.
Probemos esa regla y midamos su **accuracy** (porcentaje de aciertos).

In [4]:
# Regla: si tiene antecedentes familiares → predigo "Yes", si no → "No"
prediccion = datos["family_history"]          # ya contiene "Yes" / "No"

aciertos = prediccion == datos["treatment"]   # True cuando la regla acierta
print(f"Presicion de la regla: {aciertos.mean():.1%}")

Presicion de la regla: 68.3%


**68.3%.** Bastante mejor que el baseline (50.6%). La regla humana sí captura algo.

Fíjate en el truco de la línea `aciertos.mean()`: es el mismo que usaste en el notebook 0.1.
Promediar True/False da la **proporción de True**.

### ✍️ Ejercicio 2 · inventa tu propia regla
1. Elige otra columna de la tabla de la sección 2 y escribe una regla con ella.
2. Mide su accuracy.
3. ¿Le gana a `family_history`? ¿Le gana al baseline?

Pista: si la columna tiene valores distintos de Yes/No (como `Don't know`), puedes convertirla así:
`(datos["columna"] == "Yes").map({True: "Yes", False: "No"})`

In [5]:
# tu código aquí
es_yes = datos["family_history"] == "Yes" 
prediccion = es_yes.map({True: "Yes", False: "No"})
aciertos = prediccion == datos["treatment"]
print(f"Accuracy con benefits: {aciertos.mean():.1%}")

aciertos

Accuracy con benefits: 68.3%


0       False
1        True
2        True
3        True
4        True
        ...  
1254    False
1255     True
1256     True
1257     True
1258     True
Length: 1259, dtype: bool

<details><summary><b>Ver solución</b> (inténtalo al menos 5 minutos antes)</summary>

Ejemplo con `care_options`: "si sabe qué opciones de atención ofrece su empresa → Yes".

```python
prediccion = (datos["care_options"] == "Yes").map({True: "Yes", False: "No"})
print(f"Accuracy con care_options: {(prediccion == datos['treatment']).mean():.1%}")   # 62.9%
```

Supera al baseline, pero no a `family_history`. Lo importante es la pregunta que te obliga a hacer:
**¿cómo sé qué regla es mejor sin probarlas todas a mano?** Eso es exactamente lo que resuelve ML.

</details>

## 6 · La máquina: que el algoritmo encuentre la regla (15 min)

Ahora no escribimos la regla. Le damos los datos a un **árbol de decisión** y él busca
la pregunta que mejor separa a quienes buscaron tratamiento de quienes no.

Pasos:
1. **Convertir texto a números**: los modelos solo entienden números. `pd.get_dummies` crea
   una columna 0/1 por cada respuesta posible (`family_history_Yes`, `family_history_No`, …).
2. **`fit`**: el modelo aprende de `X` e `y`.
3. **`predict`**: el modelo responde para cada fila.

Esta receta de dos pasos (`fit` → `predict`) es **igual en todos los modelos de scikit-learn**.
Si la entiendes aquí, la entiendes para siempre.

In [6]:
from sklearn.tree import DecisionTreeClassifier, export_text

columnas = ["family_history", "care_options", "benefits",
            "obs_consequence", "remote_work", "tech_company"]

X = pd.get_dummies(datos[columnas], dtype=int)   # texto → columnas 0/1
y = datos["treatment"]

print(X.shape)
X.head()

(1259, 14)


,family_history_No,family_history_Yes,care_options_No,care_options_Not sure,care_options_Yes,benefits_Don't know,benefits_No,benefits_Yes,obs_consequence_No,obs_consequence_Yes,remote_work_No,remote_work_Yes,tech_company_No,tech_company_Yes
0,1,0,0,1,0,0,0,1,1,0,1,0,0,1
1,1,0,1,0,0,1,0,0,1,0,1,0,1,0
2,1,0,1,0,0,0,1,0,1,0,1,0,0,1
3,0,1,0,0,1,0,1,0,0,1,1,0,0,1
4,1,0,1,0,0,0,0,1,1,0,0,1,0,1


In [9]:
# Árbol de decisión con UNA sola pregunta (max_depth=1); random_state fija la semilla
modelo = DecisionTreeClassifier(max_depth=1, random_state=0)

# Entrena: busca el mejor corte entre todas las variables de X para separar y
modelo.fit(X, y)

# Predice la clase de cada fila de X
prediccion = modelo.predict(X)

# Proporción de aciertos: compara predicción vs. real y promedia los True/False
print(f"Accuracy del árbol: {(prediccion == y).mean():.1%}")

# Muestra la regla aprendida en texto, con los nombres de las columnas
print(export_text(modelo, feature_names=list(X.columns)))

Accuracy del árbol: 68.3%
|--- family_history_Yes <= 0.50
|   |--- class: No
|--- family_history_Yes >  0.50
|   |--- class: Yes



**Mira la regla que imprimió.** De todas las preguntas posibles, el árbol eligió
`family_history_Yes`: **la misma regla que pensaste tú en la sección 5**, con el mismo 68.3%.

La diferencia es que tú la adivinaste por intuición. La máquina probó **todas** las columnas
y demostró que esa era la mejor. Con 6 columnas lo puedes hacer a mano; con 600, no.

### ✍️ Ejercicio 3 · déjalo hacer más preguntas
1. Copia la celda del modelo y cambia `max_depth` a 2. ¿Qué regla nueva aparece?
2. Prueba con `max_depth` = 3, 5 y 10. Anota el accuracy de cada uno.
3. ¿El accuracy sigue subiendo para siempre?

In [25]:
# tu código aquí
# Recorre cada profundidad máxima que quieres probar: 1, 2, 3, 5 y 10
for profundidad in [1, 2, 3, 5, 10]:

    # Crea un árbol nuevo en cada vuelta, limitado a "profundidad" niveles de preguntas
    # (random_state=0 hace que el resultado sea siempre el mismo)
    modelo = DecisionTreeClassifier(max_depth=profundidad, random_state=0)

    # Entrena el árbol con los datos: aprende las reglas que mejor separan y
    modelo.fit(X, y)

    # modelo.predict(X) → predice la clase de cada fila
    # == y             → compara con la clase real (True si acertó, False si no)
    # .mean()          → promedio de True/False = proporción de aciertos (ej: 0.683)
    # :.1%             → lo muestra como porcentaje con 1 decimal (68.3%)
    print(profundidad, f"{(modelo.predict(X) == y).mean():.1%}")

1 68.3%
2 70.4%
3 71.0%
5 72.7%
10 73.4%


<details><summary><b>Ver solución</b> (inténtalo al menos 5 minutos antes)</summary>

```python
for profundidad in [1, 2, 3, 5, 10]:
    modelo = DecisionTreeClassifier(max_depth=profundidad, random_state=0)
    modelo.fit(X, y)
    print(profundidad, f"{(modelo.predict(X) == y).mean():.1%}")
```

| max_depth | accuracy |
|---|---|
| 1 | 68.3% |
| 2 | 70.4% |
| 3 | 71.0% |
| 5 | 72.7% |
| 10 | 73.4% |

Con `max_depth=2` aparece `care_options_Yes` como segunda pregunta.

Sube, pero cada vez menos, y se estanca. **Y aquí hay una trampa enorme:** estamos midiendo el
acierto **con los mismos datos con los que el modelo aprendió**. Es como dar una prueba con las
mismas preguntas que viste en la guía de estudio: sacas buena nota, pero no prueba que entendiste.
Eso se llama **sobreajuste** y lo arreglamos en el notebook 04.

</details>

## 📝 Resumen para tu cuaderno

1. **ML = datos + respuestas → reglas.** El algoritmo descubre la regla; tú no la escribes.
2. **X** son las features (las pistas), **y** es el objetivo (lo que predices). Cada fila es una **observación**.
3. **Clasificación** si `y` es una categoría; **regresión** si es un número.
4. **Baseline** = acierto del modelo más tonto (siempre la clase común). Todo modelo debe superarlo.
5. **scikit-learn siempre es:** `modelo.fit(X, y)` → `modelo.predict(X)`.
6. ⚠️ Medir el acierto con los mismos datos de entrenamiento **engaña**. Lo corregimos en el 04.

**Próximo notebook (02):** explorar y limpiar los datos. Ya viste que `Age` tiene valores
como −1726 y que `Gender` está escrito de 49 formas distintas. Vamos a ordenar ese desorden.